# 09 · HMDA：Colab GPU 上筛选 CatBoost 参数（第 1 折）

现有 R1/R2 都是**固定学习率 0.05、1600 轮**，从未调过。这里在 GPU 上快速比较更小学习率 + 早停、不同深度、L2 正则、行抽样和 DTI 上下界。

**协议（外层验证不参与任何选择）**
- 外层：固定 seed=42 五折中的第 1 折；84,000 行训练、21,000 行验证。
- 早停候选：在外层训练部分再分层留出 15% 做早停，得到最佳轮数 *b*；然后在整个外层训练部分用 `round(b / 0.85)` 轮重新训练，再在外层验证上评分。
- `B0` 是现有 GPU R2（固定 1600 轮）的同折对照；GPU 上这一折以前是 0.90915。

**判读**：单折只用于筛选。挑出相对 B0 提升最大的 1–2 个，把 `FOLDS` 改成 `[1, 2, 3, 4, 5]`、`RUN_ONLY` 填这几个名字，再跑完整五折；五折平均提升 ≥ 0.0005 且多数折同向才采用。GPU 与 CPU 的绝对分数不同（CPU 约高 0.001），所以这里看**相对 B0 的差**，选出的参数最后用 CPU 全量训练提交（08 同样的做法）。

**操作**：Colab 选 **T4 GPU**，从上到下运行。每个（候选, 折）完成后立刻存到 Drive `artifacts/gpu_tuning/<EXPERIMENT_ID>/`；断线后从头运行可续跑。预计全部候选第 1 折共约 1.5–2.5 小时，便宜的先跑。

## 1. 环境、GPU 与配置

In [6]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import importlib
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time

try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None
if drive is not None:
    drive.mount('/content/drive')

PROJECT_DIR = Path(os.environ.get(
    'HMDA_PROJECT_DIR',
    '/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis',
)).expanduser().resolve()
PREPARED_DIR = PROJECT_DIR / 'artifacts' / 'prepared'
SEED = 42
CATBOOST_VERSION = '1.2.10'
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f'找不到项目目录：{PROJECT_DIR}；请修改 PROJECT_DIR')

try:
    installed_catboost = importlib.metadata.version('catboost')
except importlib.metadata.PackageNotFoundError:
    installed_catboost = None
if installed_catboost != CATBOOST_VERSION:
    if 'catboost' in sys.modules:
        raise RuntimeError('当前运行时已经导入另一版 CatBoost；请重启 Colab 运行时后从头运行')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        f'catboost=={CATBOOST_VERSION}',
    ])
    importlib.invalidate_caches()

import catboost
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

if catboost.__version__ != CATBOOST_VERSION:
    raise RuntimeError(f'需要 CatBoost {CATBOOST_VERSION}；实际为 {catboost.__version__}')

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('未找到 nvidia-smi：请在 Colab 更改运行时类型，选择 T4 GPU 后从头运行')
gpu_probe = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    text=True, capture_output=True, check=False,
)
if gpu_probe.returncode != 0 or not gpu_probe.stdout.strip():
    raise RuntimeError('NVIDIA GPU 不可用：请在 Colab 选择 T4 GPU 后从头运行')
GPU_NAME = gpu_probe.stdout.strip().splitlines()[0]

EXPERIMENT_ID = 'catboost_screen_v1'
FOLDS = [1, 2, 3, 4, 5]     # 第二轮：五折复核；只做第 1 折筛选时改回 [1]
RUN_ONLY = ['B0_R2_fixed1600', 'C7_d8_lr03_l2_10_sub80',
            'C10_d8_lr03_l2_30_sub80', 'C11_d6_lr03_l2_10_sub80']  # None 表示全部候选
INNER_STOP_FRACTION = 0.15
MAX_ITERATIONS = 10000
OD_WAIT = 300
OUTPUT_DIR = PROJECT_DIR / 'artifacts' / 'gpu_tuning' / EXPERIMENT_ID

GPU_BASE = dict(
    loss_function='Logloss', nan_mode='Min', random_seed=SEED,
    task_type='GPU', devices='0', thread_count=4,
    allow_writing_files=False, verbose=False,
)
# early_stop=False 表示固定轮数、不早停；按预计耗时从小到大排列
CANDIDATES = {
    'B0_R2_fixed1600':   dict(depth=8, learning_rate=0.05, l2_leaf_reg=3, early_stop=False, iterations=1600),
    'C1_d8_lr05_l2_3_es': dict(depth=8, learning_rate=0.05, l2_leaf_reg=3),
    'C6_d6_lr03_l2_10':  dict(depth=6, learning_rate=0.03, l2_leaf_reg=10),
    'C3_d8_lr03_l2_10':  dict(depth=8, learning_rate=0.03, l2_leaf_reg=10),
    'C2_d8_lr03_l2_3':   dict(depth=8, learning_rate=0.03, l2_leaf_reg=3),
    'C4_d8_lr03_l2_30':  dict(depth=8, learning_rate=0.03, l2_leaf_reg=30),
    'C7_d8_lr03_l2_10_sub80': dict(depth=8, learning_rate=0.03, l2_leaf_reg=10,
                                   bootstrap_type='Bernoulli', subsample=0.8),
    'C8_d8_lr03_l2_10_DTI': dict(depth=8, learning_rate=0.03, l2_leaf_reg=10, dti=True),
    'C5_d10_lr03_l2_10': dict(depth=10, learning_rate=0.03, l2_leaf_reg=10),
    'C9_d8_lr02_l2_10':  dict(depth=8, learning_rate=0.02, l2_leaf_reg=10),
    # 第二轮：把第 1 折前三名（抽样、depth 6、L2=30）组合起来
    'C10_d8_lr03_l2_30_sub80': dict(depth=8, learning_rate=0.03, l2_leaf_reg=30,
                                    bootstrap_type='Bernoulli', subsample=0.8),
    'C11_d6_lr03_l2_10_sub80': dict(depth=6, learning_rate=0.03, l2_leaf_reg=10,
                                    bootstrap_type='Bernoulli', subsample=0.8),
}
if RUN_ONLY is not None:
    unknown = set(RUN_ONLY) - set(CANDIDATES)
    if unknown:
        raise ValueError(f'RUN_ONLY 中有未知候选：{sorted(unknown)}')
print(f'项目：{PROJECT_DIR}')
print(f'GPU：{GPU_NAME}；CatBoost {catboost.__version__}；折：{FOLDS}；输出：{OUTPUT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
项目：/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis
GPU：Tesla T4, 15360 MiB；CatBoost 1.2.10；折：[1, 2, 3, 4, 5]；输出：/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis/artifacts/gpu_tuning/catboost_screen_v1


## 2. 回读并核对 01 的数据契约（与 04 相同）

In [7]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


manifest_path = PREPARED_DIR / 'manifest.json'
schema_path = PREPARED_DIR / 'schema.json'
if not manifest_path.is_file() or not schema_path.is_file():
    raise FileNotFoundError(
        f'缺少 {manifest_path} 或 {schema_path}；请先运行 01_hmda_data_preparation.ipynb'
    )
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
schema = json.loads(schema_path.read_text(encoding='utf-8'))
require(manifest['schema_version'] == schema['schema_version'] == '1.0', '数据契约版本不匹配')
require(manifest['seed'] == SEED and manifest['n_splits'] == 5, '五折配置不匹配')
require(set(manifest['source_files']) == {'train.csv', 'test.csv', 'sample_submission.csv'},
        '原始文件清单不完整')
expected_prepared = {
    'train_features.csv', 'test_features.csv', 'train_labels.csv',
    'test_ids.csv', 'folds.csv', 'audit.csv', 'schema.json',
}
require(set(manifest['output_files']) == expected_prepared, '准备文件清单不完整')
for name, info in manifest['source_files'].items():
    path = PROJECT_DIR / name
    require(path.is_file(), f'缺少原始数据：{path}')
    require(sha256_file(path) == info['sha256'], f'原始文件已变化：{name}；请先重跑 01')
for name, info in manifest['output_files'].items():
    path = PREPARED_DIR / name
    require(path.is_file(), f'缺少准备文件：{path}')
    require(sha256_file(path) == info['sha256'], f'准备文件哈希不匹配：{name}；请先重跑 01')

FEATURE_COLUMNS = schema['feature_columns']
NUMERIC_COLUMNS = schema['numeric_columns']
CATEGORICAL_COLUMNS = schema['categorical_columns']
require(len(FEATURE_COLUMNS) == 56 and len(NUMERIC_COLUMNS) == 13
        and len(CATEGORICAL_COLUMNS) == 43, '预期为 56 列：13 数值、43 类别')
require(set(FEATURE_COLUMNS) == set(NUMERIC_COLUMNS + CATEGORICAL_COLUMNS),
        '数值/类别列分组不完整')
require(not {'id', 'accepted'} & set(FEATURE_COLUMNS), 'ID 或标签不能作为特征')


def read_features(path):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    require(frame.columns.tolist() == FEATURE_COLUMNS, f'特征列或顺序不匹配：{path.name}')
    for column in NUMERIC_COLUMNS:
        frame[column] = pd.to_numeric(
            frame[column].replace('', np.nan), errors='raise',
        ).astype('float64')
    require(not np.isinf(frame[NUMERIC_COLUMNS].to_numpy()).any(), '数值列含无穷大')
    require(not frame[CATEGORICAL_COLUMNS].eq('').any().any(), '类别列含空字符串')
    return frame


X_base = read_features(PREPARED_DIR / 'train_features.csv')
X_test_base = read_features(PREPARED_DIR / 'test_features.csv')
labels = pd.read_csv(PREPARED_DIR / 'train_labels.csv', dtype=str, keep_default_na=False)
test_id_frame = pd.read_csv(PREPARED_DIR / 'test_ids.csv', dtype=str, keep_default_na=False)
fold_frame = pd.read_csv(PREPARED_DIR / 'folds.csv', dtype=str, keep_default_na=False)
sample_submission = pd.read_csv(PROJECT_DIR / 'sample_submission.csv', dtype=str, keep_default_na=False)

require(labels.columns.tolist() == ['id', 'accepted'], '训练标签列不匹配')
require(test_id_frame.columns.tolist() == ['id'], '测试 ID 列不匹配')
require(fold_frame.columns.tolist() == ['id', 'fold'], '折号文件列不匹配')
require(sample_submission.columns.tolist() == ['id', 'accepted'], '提交模板列不匹配')
require(len(X_base) == len(labels) == len(fold_frame) == 105000, '训练行数不匹配')
require(len(X_test_base) == len(test_id_frame) == len(sample_submission) == 45000,
        '测试行数不匹配')

train_ids = labels['id']
test_ids = test_id_frame['id']
require(train_ids.is_unique and test_ids.is_unique, 'ID 重复')
require(train_ids.ne('').all() and test_ids.ne('').all(), 'ID 为空')
require(train_ids.equals(fold_frame['id']), '折号 ID 与训练行错位')
require(test_ids.equals(sample_submission['id']), '测试 ID 与提交模板顺序不符')
require(set(train_ids).isdisjoint(test_ids), '训练与测试 ID 重叠')
require(set(labels['accepted']) == {'0', '1'}, 'accepted 需同时包含 0 和 1')
raw_train_identity = pd.read_csv(
    PROJECT_DIR / 'train.csv', usecols=['id', 'accepted'],
    dtype=str, keep_default_na=False,
)
raw_test_identity = pd.read_csv(
    PROJECT_DIR / 'test.csv', usecols=['id'], dtype=str, keep_default_na=False,
)
require(train_ids.equals(raw_train_identity['id'])
        and labels['accepted'].equals(raw_train_identity['accepted']),
        '准备标签与原始 train.csv 的 ID/标签不一致')
require(test_ids.equals(raw_test_identity['id']),
        '准备测试 ID 与原始 test.csv 不一致')
y = labels['accepted'].astype('int8')
require(set(fold_frame['fold']) == {'1', '2', '3', '4', '5'}, '折号应为 1..5')
fold_assignment = fold_frame['fold'].astype('int8').to_numpy()

expected_folds = np.zeros(len(X_base), dtype=np.int8)
splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold, (_, valid_index) in enumerate(splitter.split(X_base, y), start=1):
    expected_folds[valid_index] = fold
require(np.array_equal(fold_assignment, expected_folds), '折号并非固定 seed=42 五折')
DATA_HASHES = {
    'manifest.json': sha256_file(manifest_path),
    **{name: info['sha256'] for name, info in manifest['output_files'].items()},
}
print(f'数据核对通过：训练 {len(X_base):,}，测试 {len(X_test_base):,}，56 个原始特征。')
display(pd.crosstab(fold_assignment, y, rownames=['fold'], colnames=['accepted']))

def checked_probabilities(model, features):
    require(list(model.classes_) == [0, 1], '类别顺序不是 [0, 1]')
    probability = np.asarray(model.predict_proba(features)[:, 1], dtype=float)
    require(probability.ndim == 1 and len(probability) == len(features)
            and np.isfinite(probability).all()
            and ((probability >= 0) & (probability <= 1)).all(),
            '预测概率无效')
    return probability


def write_json(path, payload):
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


def fixed_config(output_dir, config):
    """首次运行写入配置；之后配置必须完全相同，避免把不同实验的结果混在一个目录。"""
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / 'config.json'
    if path.is_file():
        saved = json.loads(path.read_text(encoding='utf-8'))
        require(saved == config,
                f'{output_dir.name} 已有不同配置的结果；改动配置时请改 EXPERIMENT_ID')
    else:
        write_json(path, config)

数据核对通过：训练 105,000，测试 45,000，56 个原始特征。


accepted,0,1
fold,,
1,3151,17849
2,3151,17849
3,3151,17849
4,3151,17849
5,3152,17848


## 3. DTI 上下界（与 04 相同，只给 `dti=True` 的候选使用）

In [8]:
DTI_BOUNDS = {
    **{str(value): (float(value), float(value)) for value in range(36, 50)},
    '20%-<30%': (20.0, 30.0),
    '30%-<36%': (30.0, 36.0),
    '50%-60%': (50.0, 60.0),
    '<20%': (np.nan, 20.0),
    '>60%': (60.0, np.nan),
    'MISSING': (np.nan, np.nan),
}


def add_dti_bounds(frame):
    unknown = set(frame['debt_to_income_ratio'].unique()) - set(DTI_BOUNDS)
    require(not unknown, f'DTI 出现未知类别：{sorted(map(str, unknown))}')
    result = frame.copy()
    pairs = frame['debt_to_income_ratio'].map(DTI_BOUNDS)
    result['dti_lower'] = pairs.map(lambda pair: pair[0]).astype('float64')
    result['dti_upper'] = pairs.map(lambda pair: pair[1]).astype('float64')
    require(result.columns.tolist() == FEATURE_COLUMNS + ['dti_lower', 'dti_upper'],
            'DTI 特征列或顺序错误')
    require(result['debt_to_income_ratio'].equals(frame['debt_to_income_ratio']),
            '原始 DTI 类别被修改')
    return result


X_dti = add_dti_bounds(X_base)
X_test_dti = add_dti_bounds(X_test_base)
print(f'候选特征：原始 56 列 + 2 列 DTI 上下界 = {X_dti.shape[1]} 列。')
display(X_dti[['debt_to_income_ratio', 'dti_lower', 'dti_upper']].head())

候选特征：原始 56 列 + 2 列 DTI 上下界 = 58 列。


,debt_to_income_ratio,dti_lower,dti_upper
0,20%-<30%,20.0,30.0
1,49,49.0,49.0
2,20%-<30%,20.0,30.0
3,44,44.0,44.0
4,<20%,NaN,20.0


## 4. 训练与评分（可续跑）

每个（候选, 折）结果写为 `<候选>__fold<k>.json`，验证概率写为同名 `.csv`。配置里只记录候选定义，所以**以后新增候选不会影响已完成结果**；但修改已有候选的参数时必须改名或改 `EXPERIMENT_ID`。

In [ ]:
fixed_config(OUTPUT_DIR, {
    'purpose': 'GPU CatBoost screening, inner 15% early stop then outer refit',
    'seed': SEED, 'inner_stop_fraction': INNER_STOP_FRACTION,
    'max_iterations': MAX_ITERATIONS, 'od_wait': OD_WAIT,
    'gpu_base': {k: v for k, v in GPU_BASE.items() if k != 'thread_count'},
    'data_hashes': DATA_HASHES,
})


def candidate_params(spec, iterations):
    params = {k: v for k, v in spec.items() if k not in {'early_stop', 'dti', 'iterations'}}
    return dict(GPU_BASE, **params, iterations=iterations)


def run_one(name, spec, fold):
    features = X_dti if spec.get('dti') else X_base
    fit_idx = np.flatnonzero(fold_assignment != fold)
    valid_idx = np.flatnonzero(fold_assignment == fold)
    started = time.perf_counter()
    best_iteration = None
    early_stopped = None
    if spec.get('early_stop', True):
        inner_fit, inner_stop = train_test_split(
            fit_idx, test_size=INNER_STOP_FRACTION, random_state=SEED,
            stratify=y.iloc[fit_idx],
        )
        probe = catboost.CatBoostClassifier(**candidate_params(spec, MAX_ITERATIONS),
                                            eval_metric='Logloss', od_type='Iter',
                                            od_wait=OD_WAIT, use_best_model=True)
        probe.fit(features.iloc[inner_fit], y.iloc[inner_fit],
                  cat_features=CATEGORICAL_COLUMNS,
                  eval_set=(features.iloc[inner_stop], y.iloc[inner_stop]))
        best_iteration = int(probe.get_best_iteration()) + 1
        early_stopped = int(probe.tree_count_) < MAX_ITERATIONS
        if not early_stopped:
            print(f'  注意：{name} 到 {MAX_ITERATIONS} 轮仍未早停，结果可能偏保守')
        rounds = int(round(best_iteration / (1 - INNER_STOP_FRACTION)))
        del probe
        gc.collect()
    else:
        rounds = int(spec['iterations'])
    model = catboost.CatBoostClassifier(**candidate_params(spec, rounds), use_best_model=False)
    model.fit(features.iloc[fit_idx], y.iloc[fit_idx], cat_features=CATEGORICAL_COLUMNS)
    require(int(model.tree_count_) == rounds, '外层重训轮数不符')
    probability = checked_probabilities(model, features.iloc[valid_idx])
    actual = y.iloc[valid_idx]
    pd.DataFrame({'id': train_ids.iloc[valid_idx].to_numpy(), 'accepted': actual.to_numpy(),
                  'prob': probability}).to_csv(
        OUTPUT_DIR / f'{name}__fold{fold}.csv', index=False, float_format='%.12f')
    result = {
        'candidate': name, 'fold': fold,
        'spec': {k: v for k, v in spec.items()},
        'feature_count': features.shape[1],
        'inner_best_iteration': best_iteration, 'early_stopped': early_stopped, 'refit_rounds': rounds,
        'valid_auc': roc_auc_score(actual, probability),
        'valid_logloss': log_loss(actual, probability, labels=[0, 1]),
        'valid_brier': brier_score_loss(actual, probability),
        'seconds': time.perf_counter() - started, 'gpu_name': GPU_NAME,
        'finished_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    write_json(OUTPUT_DIR / f'{name}__fold{fold}.json', result)
    del model
    gc.collect()
    return result


def load_results():
    rows = [json.loads(p.read_text(encoding='utf-8'))
            for p in sorted(OUTPUT_DIR.glob('*__fold*.json'))]
    return pd.DataFrame(rows)


selected = {n: s for n, s in CANDIDATES.items() if RUN_ONLY is None or n in RUN_ONLY}
for fold in FOLDS:
    for name, spec in selected.items():
        result_path = OUTPUT_DIR / f'{name}__fold{fold}.json'
        if result_path.is_file():
            saved = json.loads(result_path.read_text(encoding='utf-8'))
            require(saved['spec'] == spec, f'{name} 的定义已改变；请改候选名')
            print(f'{name} fold {fold}: 已完成 AUC={saved["valid_auc"]:.6f}，跳过')
            continue
        result = run_one(name, spec, fold)
        print(f'{name} fold {fold}: AUC={result["valid_auc"]:.6f}  '
              f'轮数={result["refit_rounds"]}  {result["seconds"]/60:.1f} 分钟', flush=True)

B0_R2_fixed1600 fold 1: 已完成 AUC=0.909209，跳过
C7_d8_lr03_l2_10_sub80 fold 1: 已完成 AUC=0.910596，跳过
C10_d8_lr03_l2_30_sub80 fold 1: AUC=0.910650  轮数=3800  8.5 分钟
C11_d6_lr03_l2_10_sub80 fold 1: AUC=0.911277  轮数=4891  7.5 分钟
B0_R2_fixed1600 fold 2: AUC=0.923237  轮数=1600  1.8 分钟
C7_d8_lr03_l2_10_sub80 fold 2: AUC=0.925117  轮数=2591  5.7 分钟
C10_d8_lr03_l2_30_sub80 fold 2: AUC=0.924317  轮数=4007  8.7 分钟
C11_d6_lr03_l2_10_sub80 fold 2: AUC=0.925117  轮数=4909  7.6 分钟
B0_R2_fixed1600 fold 3: AUC=0.916840  轮数=1600  1.8 分钟
C7_d8_lr03_l2_10_sub80 fold 3: AUC=0.918249  轮数=3144  7.1 分钟
C10_d8_lr03_l2_30_sub80 fold 3: AUC=0.917689  轮数=3938  8.9 分钟
C11_d6_lr03_l2_10_sub80 fold 3: AUC=0.917148  轮数=5920  9.2 分钟
B0_R2_fixed1600 fold 4: AUC=0.910038  轮数=1600  1.8 分钟
C7_d8_lr03_l2_10_sub80 fold 4: AUC=0.911991  轮数=2565  5.7 分钟
C10_d8_lr03_l2_30_sub80 fold 4: AUC=0.911151  轮数=4264  9.4 分钟


## 5. 与 B0 配对比较

`delta_vs_B0` 是同一折上的 AUC 差。只有同时跑了相同折的候选才会出现在五折汇总里。

In [ ]:
results = load_results()
require(not results.empty, '还没有完成的结果')
auc = results.pivot(index='fold', columns='candidate', values='valid_auc')
require('B0_R2_fixed1600' in auc.columns, '缺少对照 B0_R2_fixed1600')
delta = auc.sub(auc['B0_R2_fixed1600'], axis=0)
table = pd.DataFrame({
    'folds_done': auc.notna().sum(),
    'mean_auc': auc.mean(),
    'mean_delta_vs_B0': delta.mean(),
    'folds_better_than_B0': (delta > 0).sum(),
    'mean_refit_rounds': results.groupby('candidate')['refit_rounds'].mean(),
    'mean_logloss': results.groupby('candidate')['valid_logloss'].mean(),
    'total_minutes': results.groupby('candidate')['seconds'].sum() / 60,
}).sort_values('mean_delta_vs_B0', ascending=False)
table.to_csv(OUTPUT_DIR / 'summary.csv')
display(table.round(6))
display(delta.round(6))